# **1**. ***`Introduction`***

# 🚢 ***Task 7: Titanic Survival Prediction – ML Pipeline***

## ***Introduction***

***In this task, I worked with the Titanic dataset to build a clean and reusable Machine Learning pipeline for predicting passenger survival. I performed data preprocessing by handling missing values and separating numerical and categorical features. I used ColumnTransformer to apply StandardScaler to numerical features and OneHotEncoder to categorical features, and then combined the preprocessing steps with Logistic Regression using a single Pipeline. I also created two engineered features, FamilySize and IsAlone, and compared the model performance with and without these features. Finally, I evaluated the pipeline using Accuracy, Precision, Recall, and F1-score and saved the final trained pipeline using Joblib for future use.***

---

# **2**. ***`Import Libraries`***

In [52]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# **3**. ***`Load Dataset`***

In [53]:
df = pd.read_csv("Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# **4**. ***`Dataset Structure`***

# “***Drop Unnecessary Columns***”

In [54]:
df.drop(['PassengerId','Name','Ticket'],axis= 1, inplace= True)

In [55]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,0,3,male,22.0,1,0,7.2500,NaN,S
1,1,1,female,38.0,1,0,71.2833,C85,C
2,1,3,female,26.0,0,0,7.9250,NaN,S
3,1,1,female,35.0,1,0,53.1000,C123,S
4,0,3,male,35.0,0,0,8.0500,NaN,S


# “***Shape***”

In [56]:
df.shape

(891, 9)

# “***Information***”

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   SibSp     891 non-null    int64  
 5   Parch     891 non-null    int64  
 6   Fare      891 non-null    float64
 7   Cabin     204 non-null    object 
 8   Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(3)
memory usage: 62.8+ KB


# “***Columns***”

In [58]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin',
       'Embarked'],
      dtype='object')

# “***Statistical Summary***”

In [59]:
df.describe()

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


# “***Check Null Values***”

In [60]:
df.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Cabin       687
Embarked      2
dtype: int64

# **5**. ***`Feature Engineering`***

In [61]:
data = df.copy()

data["FamilySize"] = data["SibSp"] + data["Parch"] + 1

data["IsAlone"] = (data["FamilySize"] == 1).astype(int)

data[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


# “***Features & Target***”

In [62]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone"]

X = data[features]
y = data["Survived"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']

Target:
Survived


# **6**. ***`Train Test Split`***

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

print("Training Data:", X_train.shape)
print("Testing Data :", X_test.shape)

Training Data: (712, 9)
Testing Data : (179, 9)


# **7**. ***`Manual Approach`***

# “***Numerical & Categorical Columns***”

In [64]:
numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]

categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]

print("Numerical Columns:")
print(numerical_features)

print("\nCategorical Columns:")
print(categorical_features)

Numerical Columns:
['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']

Categorical Columns:
['Pclass', 'Sex', 'Embarked']


# **8**. ***`Manual Missing Values Handling`***

In [65]:
X_train_manual = X_train.copy()
X_test_manual = X_test.copy()

for col in numerical_features:
    median_value = X_train_manual[col].median()

    X_train_manual[col] = X_train_manual[col].fillna(median_value)
    X_test_manual[col] = X_test_manual[col].fillna(median_value)

for col in categorical_features:
    mode_value = X_train_manual[col].mode()[0]

    X_train_manual[col] = X_train_manual[col].fillna(mode_value)
    X_test_manual[col] = X_test_manual[col].fillna(mode_value)

print("✅ Missing values handled manually.")

✅ Missing values handled manually.


# **9**. ***`Manual One-Hot Encoding`***

In [66]:
X_train_manual = pd.get_dummies(
    X_train_manual,
    columns=categorical_features,
    drop_first=False
)

X_test_manual = pd.get_dummies(
    X_test_manual,
    columns=categorical_features,
    drop_first=False
)

X_test_manual = X_test_manual.reindex(
    columns=X_train_manual.columns,
    fill_value=0
)

print("Train Shape:", X_train_manual.shape)
print("Test Shape :", X_test_manual.shape)

Train Shape: (712, 14)
Test Shape : (179, 14)


# **10**. ***`Manual Scaling`***

In [67]:
scaler_manual = StandardScaler()

X_train_manual[numerical_features] = scaler_manual.fit_transform(
    X_train_manual[numerical_features]
)

X_test_manual[numerical_features] = scaler_manual.transform(
    X_test_manual[numerical_features]
)

print("✅ Numerical features scaled manually.")

✅ Numerical features scaled manually.


# **11**. ***`Train Manual Model`***

In [68]:
manual_model = LogisticRegression(max_iter=1000)

manual_model.fit(
    X_train_manual,
    y_train
)

manual_pred = manual_model.predict(X_test_manual)

print("✅ Manual Logistic Regression trained.")

✅ Manual Logistic Regression trained.


# **12**. ***`Manual Evaluation`***

In [69]:
manual_accuracy = accuracy_score(y_test, manual_pred)
manual_precision = precision_score(y_test, manual_pred)
manual_recall = recall_score(y_test, manual_pred)
manual_f1 = f1_score(y_test, manual_pred)

print("========== MANUAL APPROACH ==========")
print(f"Accuracy  : {manual_accuracy:.4f}")
print(f"Precision : {manual_precision:.4f}")
print(f"Recall    : {manual_recall:.4f}")
print(f"F1 Score  : {manual_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, manual_pred))

========== MANUAL APPROACH ==========
Accuracy  : 0.8156
Precision : 0.8103
Recall    : 0.6812
F1 Score  : 0.7402

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179



# **13**. ***`Pipeline Approach`***

# “***Numerical Features***”

In [70]:
numerical_features = [
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize",
    "IsAlone"
]
numerical_features

['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']

# “***Categorical Features***”

In [71]:
categorical_features = [
    "Pclass",
    "Sex",
    "Embarked"
]
categorical_features

['Pclass', 'Sex', 'Embarked']

# “***Numerical Preprocessing***”

In [72]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("✅ Numerical transformer created.")

✅ Numerical transformer created.


# “***Categorical Preprocessing***”

In [73]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

print("✅ Categorical transformer created.")

✅ Categorical transformer created.


# **14**. ***`Column Transformer`***

In [74]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("✅ ColumnTransformer created successfully.")

✅ ColumnTransformer created successfully.


# **15**. ***`Complete Pipeline`***

In [75]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

# “***Train Pipeline***”

In [76]:
pipeline.fit(X_train, y_train)

print("✅ Pipeline trained successfully!")

✅ Pipeline trained successfully!


# “***Pipeline Predictions***”

In [77]:
pipeline_pred = pipeline.predict(X_test)

print("Predictions generated successfully.")
print(pipeline_pred[:10])

Predictions generated successfully.
[0 0 0 0 1 1 1 0 0 0]


# “***Precision, Recall & F1***”

In [79]:
pipeline_accuracy = accuracy_score(y_test, pipeline_pred)
pipeline_precision = precision_score(y_test, pipeline_pred)
pipeline_recall = recall_score(y_test, pipeline_pred)
pipeline_f1 = f1_score(y_test, pipeline_pred)

print("========== PIPELINE APPROACH ==========")
print(f"Accuracy  : {pipeline_accuracy:.4f}")
print(f"Precision : {pipeline_precision:.4f}")
print(f"Recall    : {pipeline_recall:.4f}")
print(f"F1 Score  : {pipeline_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, pipeline_pred))

========== PIPELINE APPROACH ==========
Accuracy  : 0.8156
Precision : 0.8103
Recall    : 0.6812
F1 Score  : 0.7402

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179



# **16**. ***`Compare MANUAL VS PIPELINE`***

In [92]:
comparison = pd.DataFrame({
    "Approach": [
        "Manual Preprocessing",
        "ML Pipeline"
    ],
    "Accuracy": [
        manual_accuracy,
        pipeline_accuracy
    ],
    "Precision": [
        manual_precision,
        pipeline_precision
    ],
    "Recall": [
        manual_recall,
        pipeline_recall
    ],
    "F1 Score": [
        manual_f1,
        pipeline_f1
    ]
})

comparison

,Approach,Accuracy,Precision,Recall,F1 Score
0,Manual Preprocessing,0.815642,0.810345,0.681159,0.740157
1,ML Pipeline,0.815642,0.810345,0.681159,0.740157


# **17**. ***`Confirm Same or Better Result`***

In [93]:
print(f"Manual Accuracy   : {manual_accuracy:.4f}")
print(f"Pipeline Accuracy : {pipeline_accuracy:.4f}")

if pipeline_accuracy >= manual_accuracy:
    print("\n✅ Pipeline gives the same or better accuracy than the manual approach.")
else:
    print("\n⚠️ Manual approach has slightly higher accuracy.")

print("\nThe Pipeline approach is more reusable, consistent, and easier to maintain.")

Manual Accuracy   : 0.8156
Pipeline Accuracy : 0.8156

✅ Pipeline gives the same or better accuracy than the manual approach.

The Pipeline approach is more reusable, consistent, and easier to maintain.


## ***🔵 Feature Engineering Comparison***
***The task also requires checking whether adding FamilySize and IsAlone features improves the model’s performance compared to the original features.***

# **18**. ***`Original Features Without Engineered Features`***

In [82]:
original_features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked"
]

X_original = data[original_features]

# **19**. ***`Split Original Data`***

In [83]:
X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(
    X_original,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# **20**. ***`Original Preprocessor`***

In [84]:
original_numeric = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

original_categorical = [
    "Pclass",
    "Sex",
    "Embarked"
]

original_preprocessor = ColumnTransformer(
    transformers=[
        ("num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            original_numeric
        ),
        ("cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            original_categorical
        )
    ]
)

# **21**. ***`Original Pipeline`***

In [85]:
original_pipeline = Pipeline(
    steps=[
        ("preprocessor", original_preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

original_pipeline.fit(
    X_train_original,
    y_train_original
)

original_pred = original_pipeline.predict(X_test_original)

original_accuracy = accuracy_score(
    y_test_original,
    original_pred
)

print(f"Accuracy Without Engineered Features: {original_accuracy:.4f}")

Accuracy Without Engineered Features: 0.8045


# **22**. ***`Feature Engineering Comparison`***

In [94]:
feature_comparison = pd.DataFrame({
    "Feature Set": [
        "Original Features",
        "Original + FamilySize + IsAlone"
    ],
    "Accuracy": [
        original_accuracy,
        pipeline_accuracy
    ]
})

feature_comparison

,Feature Set,Accuracy
0,Original Features,0.804469
1,Original + FamilySize + IsAlone,0.815642


# **23**. ***`Check Improvements`***

In [95]:
feature_improvement = pipeline_accuracy - original_accuracy

print(f"Accuracy Improvement: {feature_improvement:.4f}")

if feature_improvement > 0:
    print("✅ FamilySize and IsAlone improved model performance.")
elif feature_improvement < 0:
    print("⚠️ Engineered features slightly decreased accuracy.")
else:
    print("➡️ Engineered features produced the same accuracy.")

Accuracy Improvement: 0.0112
✅ FamilySize and IsAlone improved model performance.


# **24**. ***`Final Metrics`***

In [96]:
final_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Score": [
        pipeline_accuracy,
        pipeline_precision,
        pipeline_recall,
        pipeline_f1
    ]
})

final_results

,Metric,Score
0,Accuracy,0.815642
1,Precision,0.810345
2,Recall,0.681159
3,F1 Score,0.740157


# **25**. ***`Save Pipeline`***

# “***Using Joblib***”

In [88]:
joblib.dump(
    pipeline,
    "titanic_final_pipeline.pkl"
)

print("✅ Final pipeline saved successfully!")
print("File: titanic_final_pipeline.pkl")

✅ Final pipeline saved successfully!
File: titanic_final_pipeline.pkl


# **26**. ***`Load Pipeline`***

In [89]:
loaded_pipeline = joblib.load(
    "titanic_final_pipeline.pkl"
)

print("✅ Pipeline loaded successfully!")

✅ Pipeline loaded successfully!


# **27**. ***`Test Saved Pipeline`***

In [90]:
loaded_predictions = loaded_pipeline.predict(X_test)

loaded_accuracy = accuracy_score(
    y_test,
    loaded_predictions
)

print(f"Accuracy of Saved Pipeline: {loaded_accuracy:.4f}")

Accuracy of Saved Pipeline: 0.8156


# **28**. ***`Final Summary`***

In [101]:
print("=" * 55)
print("🚢 TITANIC SURVIVAL PREDICTION - TASK 7")
print("=" * 55)

print("\n📌 Manual Approach")
print(f"Accuracy  : {manual_accuracy:.4f}")
print(f"Precision : {manual_precision:.4f}")
print(f"Recall    : {manual_recall:.4f}")
print(f"F1 Score  : {manual_f1:.4f}")

print("\n📌 Pipeline Approach")
print(f"Accuracy  : {pipeline_accuracy:.4f}")
print(f"Precision : {pipeline_precision:.4f}")
print(f"Recall    : {pipeline_recall:.4f}")
print(f"F1 Score  : {pipeline_f1:.4f}")
print("Improvement: Pipeline gives the same or better accuracy than the manual approach.")

print("\n📌 Feature Engineering")
print(f"Without Engineered Features : {original_accuracy:.4f}")
print(f"With FamilySize + IsAlone   : {pipeline_accuracy:.4f}")
print(f"Improvement                 : {feature_improvement:.4f}")

print("\n📌 Final Pipeline")
print("Saved as: titanic_final_pipeline.pkl")

print("\n🎉 TASK 7 COMPLETED SUCCESSFULLY!")

🚢 TITANIC SURVIVAL PREDICTION - TASK 7

📌 Manual Approach
Accuracy  : 0.8156
Precision : 0.8103
Recall    : 0.6812
F1 Score  : 0.7402

📌 Pipeline Approach
Accuracy  : 0.8156
Precision : 0.8103
Recall    : 0.6812
F1 Score  : 0.7402
Improvement: Pipeline gives the same or better accuracy than the manual approach.

📌 Feature Engineering
Without Engineered Features : 0.8045
With FamilySize + IsAlone   : 0.8156
Improvement                 : 0.0112

📌 Final Pipeline
Saved as: titanic_final_pipeline.pkl

🎉 TASK 7 COMPLETED SUCCESSFULLY!


## **29**. ***`Conclusion`***

***In this task, I successfully built a clean and reusable Machine Learning pipeline for Titanic Survival Prediction using Scikit-learn. The Pipeline combined data preprocessing with Logistic Regression, using StandardScaler for numerical features and OneHotEncoder for categorical features through ColumnTransformer. I also created two engineered features, FamilySize and IsAlone, which improved the model accuracy from 80.45% to 81.56%. The Pipeline achieved the same performance as the manual approach, with an accuracy of 81.56%. Finally, I saved the trained Pipeline using Joblib and successfully loaded and tested it, demonstrating that the complete workflow can be reused consistently for future predictions.***

---